# matmul-back-transpose-pair — worked example 1: Derive the Matmul Backward Shapes from First Principles

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matmul-back-transpose-pair`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

For `out = x @ y` with `x: (m, k)`, `y: (k, n)`, `out: (m, n)`, the backward gradients must have the same shapes as the original inputs. The only matmul of `grad_out: (m, n)` that produces `(m, k)` is `grad_out @ y.T`, and the only one producing `(k, n)` is `x.T @ grad_out`. Shape compatibility uniquely determines which operand gets transposed — there is no need to memorize a formula; you derive it from dimensions.

## Worked solution

**Step 1 — recall the shapes.**
`x: (m, k)`, `y: (k, n)`, `out: (m, n)`, `grad_out: (m, n)`.

**Step 2 — derive dL/dx by shape.**
We need a `(m, k)` tensor. We have `grad_out: (m, n)` and can use `y` or `y.T`. The only matmul that produces `(m, k)` is `(m, n) @ (n, k)` = `grad_out @ y.T`. So `dL/dx = grad_out @ y.T`.

**Step 3 — derive dL/dy by shape.**
We need a `(k, n)` tensor. We have `grad_out: (m, n)` and can use `x` or `x.T`. The matmul `(k, m) @ (m, n)` = `x.T @ grad_out` produces `(k, n)`. So `dL/dy = x.T @ grad_out`.

**Step 4 — verify both gradients against autograd.**
We run the forward with `requires_grad=True`, call `.backward()`, and confirm that our manual gradients match PyTorch's.

In [ ]:
import torch as t

t.manual_seed(19)
m, k, n = 4, 3, 5
x = t.randn(m, k)
y = t.randn(k, n)
out = x @ y  # (m, n)
grad_out = t.randn(m, n)

# Manual backward using the transpose-pair rule
dL_dx = grad_out @ y.T    # (m,n)@(n,k) = (m,k)
dL_dy = x.T @ grad_out    # (k,m)@(m,n) = (k,n)

print(f"x: {x.shape}, y: {y.shape}, out: {out.shape}")
print(f"grad_out: {grad_out.shape}")
print(f"dL/dx (manual): {dL_dx.shape}  expected ({m},{k})")
print(f"dL/dy (manual): {dL_dy.shape}  expected ({k},{n})")

# Verify with autograd
x_ag = x.clone().requires_grad_(True)
y_ag = y.clone().requires_grad_(True)
out_ag = x_ag @ y_ag
out_ag.backward(grad_out)

print(f"\ndL/dx match: {t.allclose(dL_dx, x_ag.grad, atol=1e-5)}")
print(f"dL/dy match: {t.allclose(dL_dy, y_ag.grad, atol=1e-5)}")